# Python 184 — Final Review

---

## Table of Contents

1. [Files: opening, modes, reading, `with`](#1)
2. [CSV files](#2)
3. [Pickle: saving Python objects](#3)
4. [Floats vs `Decimal`](#4)
5. [Rounding modes & `round()`](#5)
6. [F-string number formatting](#6)
7. [Strings: slicing, search, immutability, predicates, `split`](#7)
8. [Datetime: classes, parsing, formatting, `timedelta`](#8)
9. [Dictionaries: lookup, methods, operators, iteration, nesting](#9)
10. [Exceptions](#10)

## Coverage

This review covers the second half of the course, Lectures 16–24: files, CSV, pickle, `Decimal` and rounding, f-string formatting, strings, `datetime`, dictionaries, and a short section on exceptions.


---
<a id='1'></a>
## 1. Files: opening, modes, reading, `with`

Open a file with `open(path, mode)`. Always close it — easiest is to use `with`, which closes the file automatically when the block ends, **even if an exception is raised**.

### File modes
| Mode | Meaning |
|------|---------|
| `"r"` | Read (default). Errors if the file does not exist. |
| `"w"` | Write. **Erases** existing contents. Creates the file if missing. |
| `"a"` | Append. Adds to the end. Creates the file if missing. |
| `"rb"` / `"wb"` | Binary read / write (used for `pickle`). |

### Why `with`?
Without it you'd write `f.close()` manually, and an exception in between could skip it. `with` is a context manager that guarantees the cleanup.

In [ ]:
# Write a file (mode "w" erases the file first if it already exists)
with open("scores.txt", "w") as f:
    f.write("Alice 92\n")
    f.write("Bob 85\n")
    f.write("Carol 78\n")
# the file is closed automatically here

### Reading line by line

Three valid ways. The first is the most memory-efficient because it reads one line at a time. `readlines()` loads the **whole file** as a list — best for small files where you also need random access (sorting, slicing, re-reading).

> Note: there is no `file.readall()` method.

In [ ]:
# 1. Iterate the file object directly
with open("scores.txt", "r") as f:
    for line in f:
        print(line.strip())

In [ ]:
# 2. .readlines() returns a LIST of lines (each still ends with \n)
with open("scores.txt", "r") as f:
    lines = f.readlines()
    print(lines)

In [ ]:

# 3. while + .readline()
with open("scores.txt", "r") as f:
    while line := f.readline():
        print(line.strip())

---
<a id='2'></a>
## 2. CSV files

CSV = Comma-Separated Values. Use the built-in `csv` module instead of splitting strings yourself — it handles quoting and embedded commas correctly.

### Why pass `newline=""`
Without it, on Windows the `csv` module can produce a blank line between every row. The `csv` module handles line endings itself.

### `csv.reader`
`csv.reader(f)` iterates a file row by row. **Each row comes back as a list of strings** — one entry per comma-separated field. You can index into it like any other list (`row[0]`, `row[1]`, …).

In [ ]:
import csv

rows = [
    ["name", "age", "city"],
    ["Alice", 30, "Portland"],
    ["Bob", 25, "Bangor"],
]

with open("people.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for row in rows:
        writer.writerow(row)

In [ ]:
import csv

# csv.reader: each row is a LIST of strings
with open("people.csv", "r", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

---
<a id='3'></a>
## 3. Pickle: saving Python objects

`pickle` saves an entire Python object (list, dict, etc.) to a binary file and loads it back exactly as it was.

| Step | Call |
|------|------|
| Save | `pickle.dump(obj, f)` |
| Load | `pickle.load(f)` |

- Open the file in **binary** mode: `"wb"` to save, `"rb"` to load.
- **Never** unpickle data from an untrusted source — it can run arbitrary code.

In [ ]:
import pickle

shelter = {
    "Rex":      {"species": "dog", "age": 4},
    "Whiskers": {"species": "cat", "age": 7},
}

# Save (wb = write binary)
with open("shelter.pkl", "wb") as f:
    pickle.dump(shelter, f)

# Load (rb = read binary)
with open("shelter.pkl", "rb") as f:
    restored = pickle.load(f)

print(restored)
print(restored == shelter)   # True

---
<a id='4'></a>
## 4. Floats vs `Decimal`

Python `float` is binary IEEE-754. Many decimal numbers can't be represented exactly:

In [ ]:
print(0.1 + 0.2)            # 0.30000000000000004
print(0.1 + 0.2 == 0.3)     # False

### Why does `Decimal` exist?

`Decimal` stores numbers in **base-10**, so `0.1` is exactly `0.1`. Reach for `Decimal` whenever you need exact decimal arithmetic — the canonical example is **money** (prices, taxes, totals), where binary rounding errors are not acceptable.

### Two ways to import
Both let you do `Decimal` arithmetic, but the access syntax is different.

| Import | How you write it |
|--------|------------------|
| `import decimal` | `decimal.Decimal("0.1")` |
| `from decimal import Decimal` | `Decimal("0.1")` (no prefix) |

> Pass numbers as **strings** (`Decimal("0.1")`)

In [ ]:
from decimal import Decimal

a = Decimal("0.1")
b = Decimal("0.2")

print(a + b)                     # 0.3
print(a + b == Decimal("0.3"))   # True

In [ ]:
# WRONG: passing a float loses precision before Decimal sees it
from decimal import Decimal
print(Decimal(0.1))   # 0.1000000000000000055511151231257827021181583404541015625

---
<a id='5'></a>
## 5. Rounding modes & `round()`

Python's built-in `round()` uses **banker's rounding** (half-even): `0.5` rounds to `0`, `1.5` rounds to `2`. `Decimal.quantize()` lets you pick the mode explicitly.

| Mode | Behavior |
|------|----------|
| `ROUND_HALF_UP` | classic — half rounds away from 0 (`0.5 → 1`) |
| `ROUND_HALF_DOWN` | half rounds toward 0 (`0.5 → 0`) |
| `ROUND_HALF_EVEN` | banker's rounding (default for `Decimal` and `round()`) |
| `ROUND_CEILING` | always toward +∞ |
| `ROUND_FLOOR` | always toward −∞ |

In [ ]:
# Built-in round() is half-EVEN
print(round(0.5))    # 0
print(round(1.5))    # 2
print(round(2.5))    # 2
print(round(3.5))    # 4

In [ ]:
from decimal import Decimal, ROUND_HALF_UP, ROUND_HALF_EVEN

x = Decimal("2.5")
print(x.quantize(Decimal("1"), rounding=ROUND_HALF_UP))     # 3
print(x.quantize(Decimal("1"), rounding=ROUND_HALF_EVEN))   # 2

---
<a id='6'></a>
## 6. F-string number formatting

The format spec lives **after a colon** inside the braces:
```
f"{value:[fill][align][width][,][.precision][type]}"
```

| Spec | Effect | Example |
|------|--------|---------|
| `:.2f` | 2 decimal places, fixed | `3.14` |
| `:,` | thousands separator | `12,345` |
| `:,.2f` | thousands AND 2 decimals | `12,345.68` |
| `:.1%` | percentage, 1 decimal | `7.5%` |
| `:>10` | right-align in width 10 | `"     hello"` |
| `:<10` | left-align in width 10 | `"hello     "` |
| `:^10` | center in width 10 | `"  hello   "` |

> Order matters: comma BEFORE the dot — `:,.2f` is correct, `:.2,f` is not. Same for the `f` type letter — it always comes last (`:.2f`, never `:2.f`).

In [ ]:
value = 12345.6789

print(f"{value:.2f}")     # 12345.68
print(f"{value:,.2f}")    # 12,345.68
print(f"{value:.0f}")     # 12346 (rounded, half-even)

In [ ]:
# Percentages: the % type multiplies by 100 and appends a percent sign
rate = 0.075
print(f"{rate:.1%}")      # 7.5%
print(f"{rate:.0%}")      # 8%

### Three valid ways to display 2 decimal places
All three are valid — pick the one that fits the situation.

In [ ]:
from decimal import Decimal

value = 12.345

print(f"{value:.2f}")                                       # 12.35  (string)
print(round(value, 2))                                       # 12.35  (float)
print(Decimal(str(value)).quantize(Decimal("0.01")))        # 12.35  (Decimal)

---
<a id='7'></a>
## 7. Strings: slicing, search, immutability, predicates, `split`

Strings are sequences — index and slice them like lists. **Slice end is exclusive.**

```
 T  h  e     s  a  l  m  o  n  i s  s w i m m i n g  u p s t r e a m
 0  1  2  3  4  5  6  7  8  9 ...                                 -1
```
`message[4:10]` returns `"salmon "` — six characters starting at index 4 (note the trailing space).

In [ ]:
message = "The salmon is swimming upstream"

print(message[0])       # 'T'
print(message[-1])      # 'm'
print(message[4:10])    # 'salmon ' (note trailing space)
print(message[:3])      # 'The'
print(message[-8:])     # 'upstream'

### Searching inside a string

| Call | Returns |
|------|---------|
| `s.find(sub)` | index of first match, or **-1** if not found |
| `s.index(sub)` | index of first match, **raises `ValueError`** if not found |
| `sub in s` | `True` / `False` (case-sensitive) |
| `s.count(sub)` | number of non-overlapping occurrences |
| `s.startswith(sub)` | `True` if the string begins with `sub` (case-sensitive) |

> The `in` operator and `startswith` are **both case-sensitive** on strings. `"Python" in "I love python"` is `False`.

In [ ]:
print("hello".find("l"))            # 2
print("hello".find("z"))            # -1  (NOT an error)

In [ ]:
print("Python" in "I love python")  # False — 'in' is case-sensitive!

In [ ]:
print("abracadabra".count("a"))     # 5

In [ ]:
print("Hello123".startswith("He"))  # True
print("Hello123".startswith("he"))  # False — case-sensitive

### Strings are immutable

String methods like `.upper()`, `.lower()`, `.replace()` always return a **new** string. The original is unchanged. To 'modify' a string you reassign: `s = s.upper()`.

In [ ]:
greeting = "hello world"
result = greeting.upper()

print(greeting)   # 'hello world'  — original unchanged
print(result)     # 'HELLO WORLD' — new string

### Predicate methods (return `True` / `False`)
All return `True` only when **every** character qualifies AND the string is non-empty.

| Method | True when |
|--------|-----------|
| `s.isdigit()` | every character is a digit |
| `s.isalpha()` | every character is a letter |
| `s.isalnum()` | every character is a letter or digit |
| `s.isupper()` | every cased character is uppercase |
| `s.islower()` | every cased character is lowercase |

In [ ]:
print("12345".isdigit())     # True
print("Hello123".isdigit())  # False — has letters
print("Hello".isalpha())     # True
print("Hello123".isalpha())  # False — has digits
print("PYTHON".isupper())    # True

### `split` and `join`

- `s.split(sep)` → splits a string into a **list** of substrings.
- `sep.join(list_of_strs)` → glues a list back into a single string.

Splitting is the standard way to tear apart a CSV line by hand: `"name,age,city".split(",")` → `["name", "age", "city"]`. (For real CSVs prefer the `csv` module, but `split` is fine for simple cases.)

In [ ]:
line = "name,age,city"
parts = line.split(",")
print(parts)                     # ['name', 'age', 'city']

print(" | ".join(parts))         # 'name | age | city'

# split() with no argument splits on ANY run of whitespace
print("  hello   world  ".split())

---
<a id='8'></a>
## 8. Datetime: classes, parsing, formatting, `timedelta`

Four classes from the `datetime` module:

| Class | Stores |
|-------|--------|
| `date` | a calendar date — year/month/day (no time-of-day) |
| `time` | a time-of-day — hour/minute/second (no date) |
| `datetime` | both combined — full timestamp |
| `timedelta` | a duration / difference (days, seconds, microseconds) — **not** a moment in time |

Standard import:
```python
from datetime import date, time, datetime, timedelta
```

> Common gotcha: `import datetime` imports the **module**; `from datetime import datetime` imports the **class** of the same name.

In [ ]:
from datetime import date, time, datetime, timedelta

today = date.today()
now = datetime.now()

print(today)
print(now)
print(now.year, now.month, now.day, now.hour, now.minute)

In [ ]:
from datetime import date, datetime

halloween = datetime(2026, 10, 31, 18, 30, 0)
graduation = date(2026, 5, 15)

print(halloween)
print(graduation)

### Parsing & formatting: `strptime` / `strftime`

| Function | Direction |
|----------|-----------|
| `datetime.strptime(text, fmt)` | **str → datetime** (parse) |
| `dt.strftime(fmt)` | **datetime → str** (format) |

Mnemonic: **strpt**ime **p**arses, **strft**ime **f**ormats.

### Common format codes
| Code | Meaning | Example |
|------|---------|---------|
| `%Y` | 4-digit year | `2026` |
| `%m` | month, zero-padded | `05` |
| `%d` | day, zero-padded | `01` |
| `%H` | hour 00-23 | `18` |
| `%I` | hour 01-12 | `06` |
| `%M` | minute | `30` |
| `%S` | second | `00` |
| `%p` | AM / PM | `PM` |
| `%A` | full weekday | `Friday` |
| `%B` | full month | `October` |

> Watch out: `%d` is **always zero-padded** (day `1` becomes `"01"`, not `"1"`).

In [ ]:
from datetime import datetime

# US-style date string — month first
dt = datetime.strptime("07/20/2026", "%m/%d/%Y")
print(dt)                              # 2026-07-20 00:00:00
print(dt.strftime("%A, %B %d, %Y"))    # Monday, July 20, 2026

In [ ]:
from datetime import datetime

halloween = datetime(2026, 10, 31, 18, 30, 0)

# 24-hour format
print(f"{halloween:%H:%M}")        # 18:30

# 12-hour with AM/PM
print(f"{halloween:%I:%M %p}")     # 06:30 PM

# Full sentence
print(f"{halloween:%A, %B %d at %I:%M %p}")

### `timedelta`: arithmetic with dates

A `timedelta` is a **duration**, not a moment. You can:

- Add or subtract one from a `date` or `datetime` to shift it.
- Subtract two `datetime`s and get a `timedelta` back.

| Attribute / method | What it gives |
|--------------------|---------------|
| `td.days` | whole-day part of the duration |
| `td.seconds` | seconds **within the current day only** (0–86399) |
| `td.total_seconds()` | the full duration as one `float` of seconds |

**Practical examples**
- *Deadline:* `due_date = today + timedelta(days=14)` — a date 14 days from today.
- *Countdown:* `time_left = launch - now` — a `timedelta` you can `.total_seconds()` to display.
- *Age:* `age = today - birthday` — a `timedelta` whose `.days` divided by 365 is years.

In [ ]:
from datetime import date, timedelta

event = date(2026, 5, 1)
print(event + timedelta(days=7))    # 2026-05-08
print(event - timedelta(days=30))   # 2026-04-01

In [ ]:
from datetime import datetime

start = datetime(2026, 5, 1, 9, 0, 0)
end   = datetime(2026, 5, 3, 17, 30, 0)

diff = end - start
print(diff)                  # 2 days, 8:30:00
print(diff.days)             # 2

In [ ]:
print(diff.seconds)          # 30600  (only the sub-day part!)
print(diff.total_seconds())  # 203400.0  (the WHOLE duration)

---
<a id='9'></a>
## 9. Dictionaries: lookup, methods, operators, iteration, nesting

A dictionary stores **key → value** pairs. Keys must be **unique** and hashable (strings, numbers, tuples). Values can be anything.

```python
shelter = {"Rex": "dog", "Whiskers": "cat"}
```

### Lookup, add, remove
| Operation | Syntax | Notes |
|-----------|--------|-------|
| Look up | `d[key]` | Raises `KeyError` if missing |
| Safe look up | `d.get(key, default)` | Returns `default` (or `None`) instead of raising |
| Add / update | `d[key] = value` | Same syntax for both — keys are unique |
| Membership | `key in d` | Checks **keys** (not values) |
| Delete | `del d[key]` or `d.pop(key)` | Both raise `KeyError` if missing |
| Empty the whole dict | `d.clear()` | Removes **every** key, not just one |

> `d.clear()` is a real dict method, but it wipes the whole dictionary, not a single key.

In [ ]:
shelter = {"Rex": "dog", "Whiskers": "cat"}

print(shelter["Rex"])                   # 'dog'
print(shelter.get("Buddy", "Unknown"))  # 'Unknown' — no KeyError

In [ ]:
shelter["Buddy"] = "dog"     # add a new key
shelter["Rex"] = "puppy"     # OVERWRITES (keys are unique)
print(shelter)

In [ ]:
print("Whiskers" in shelter)   # True

### Why `d[key]` vs `d.get(key)` matters

- `d[key]` raises `KeyError` if the key is missing — use it when the key **must** exist (a missing key is a bug worth crashing on).
- `d.get(key)` returns `None` (or your default) and **never** raises — use it when missing is normal and you want to fall back.

In [ ]:
d = {"a": 1}

# d.get() — graceful
print(d.get("missing"))            # None
print(d.get("missing", "n/a"))     # 'n/a'

# d[key] — raises
try:
    d["missing"]
except KeyError as e:
    print("KeyError:", e)

### The `|=` update operator (Python 3.9+)

`d1 |= d2` merges `d2` into `d1` in place: new keys are added, overlapping keys are **overwritten**.

In [ ]:
shelter = {"Rex": {"species": "dog"}}
new_pet = {"Whiskers": {"species": "cat"}}

shelter |= new_pet
print(shelter)

### Removing keys

In [ ]:
counts = {"apple": 3, "bread": 1, "milk": 2}

del counts["apple"]    # WORKS
counts.pop("bread")    # WORKS (also returns the removed value)
print(counts)           # {'milk': 2}

# removes EVERYTHING, not just one key:
counts.clear()
print(counts)           # {}

### Iterating a dictionary — three forms

| Loop | Loop variable holds | When to use |
|------|---------------------|-------------|
| `for key in d:` | each KEY | you only need keys (or you'll look up `d[key]` yourself) |
| `for value in d.values():` | each VALUE | the keys don't matter |
| `for k, v in d.items():` | a `(key, value)` PAIR (unpacked) | you need both at once |

> `.items()` is a **dict** method. Lists do **not** have `.items()` — to iterate a list with index/value pairs you'd use `enumerate(my_list)`.

In [ ]:
prices = {"apple": 1.20, "bread": 3.50, "milk": 2.99}

# 1) keys only
for key in prices:
    print(key)

In [ ]:
# 2) values only
for value in prices.values():
    print(value)

In [ ]:
# 3) both — typical for printing tables
for k, v in prices.items():
    print(f"{k}: {v:.2f}")

### Nested dictionaries

Values can themselves be dictionaries. Reach in with **chained subscripts**: `d[outer][inner]`.

If the outer key might be missing, you have safer options than the bare chain — they all avoid `KeyError`:

1. **Guard with `in`:** `if outer in d and inner in d[outer]: ...`
2. **`.get()` with a default value:** `d[outer].get(inner, "n/a")`
3. **Chained `.get()` with `{}`:** `d.get(outer, {}).get(inner)` — the empty dict default keeps the second `.get()` from crashing if `outer` was missing.

In [ ]:
team = {
    "Mike":    {"position": "SG", "age": 63, "status": "Active"},
    "Scottie": {"position": "SF", "age": 60, "status": "Inactive"},
    "Dennis":  {"position": "PF", "age": 64, "status": "Active"},
}

# Chained subscripts — outer key first, then inner
print(team["Dennis"]["position"])   # 'PF'
print(team["Mike"]["age"])          # 63

In [ ]:
# Add a whole new entry
team["Phil"] = {"position": "Coach", "age": 80, "status": "Active"}

# Iterate the structure
for name, info in team.items():
    print(f"{name}: {info['position']} ({info['status']})")

In [ ]:
contacts = {
    "Joel": {"city": "San Francisco", "phone": "555-555-1111"},
    "Anne": {"city": "Fresno",        "phone": "555-555-2222"},
}

# Safe versions:
print(contacts.get("Joel", {}).get("email"))           # None
print(contacts["Joel"].get("email", "n/a"))            # 'n/a'

if "email" in contacts["Joel"]:
    print(contacts["Joel"]["email"])
else:
    print("no email on file")

---
<a id='10'></a>
## 10. Exceptions

Exception handling was covered only briefly in lecture, but the structure is worth knowing.

```python
try:
    risky()
except SomeError:
    # runs only if SomeError was raised
except OtherError:
    # runs only if OtherError was raised
else:
    # runs only if NO exception was raised
finally:
    # ALWAYS runs (success, caught, or even uncaught)
```

Key facts:
- `else` runs **only on the success path** (no exception happened).
- `finally` **always** runs — whether the try succeeded, raised a caught exception, or even raised an uncaught one.
- Python's keyword is `except`, not `catch`.
- The first matching `except` wins; later ones are skipped.

In [ ]:
# Failure path: ValueError caught, else skipped, finally runs
try:
    x = int("abc")
except ValueError:
    print("bad number")
except ZeroDivisionError:
    print("div zero")
else:
    print("ok")
finally:
    print("done")
# Output:
# bad number
# done

In [ ]:
# Success path: else runs, no except runs, finally still runs
try:
    x = int("42")
except ValueError:
    print("bad number")
else:
    print("ok")
finally:
    print("done")
# Output:
# ok
# done

### Common built-in exceptions

| Exception | Raised when |
|-----------|-------------|
| `ValueError` | right type, wrong value (e.g. `int("abc")`) |
| `TypeError` | wrong type (e.g. `"a" + 1`) |
| `KeyError` | dict key not found |
| `IndexError` | list index out of range |
| `ZeroDivisionError` | division by zero |
| `FileNotFoundError` | `open("missing.txt", "r")` |
| `AttributeError` | object has no such attribute / method |